# Country Filter & Refinement

This code is included when launching `main.ipynb`

In [1]:
import pandas as pd
import os
import sys
sys.path.insert(1, r"/workspace/qgi\trojan\data engine")
import dataPrep as dp
from collections import Counter
import pickle 
import os

countriesPatternsPath = r"/workspace/qgi\trojan\countries patterns\\"
projectPath           = r"/workspace/qgi"
countriesPatternsPath = r"/workspace/qgi\trojan\countries patterns\\"
countriesFolderPath   = "/workspace/qgi\\trojan\\countries patterns\\"

## Country Filter
# Creating individual dataframes (csv) for each country by taking their patterns from each patternized index
patternized_indexes = []
for root, dirs, files in os.walk(projectPath):
    for file in files:
        if file.endswith("Index_PATTERNS_DF.csv"):
             patternized_indexes.append(os.path.join(root, file))

for path in patternized_indexes:
    dp.countryFilter(pd.read_csv(path).drop("Unnamed: 0", axis = 1), countriesPatternsPath)

## Refining
# Refining the separate country dataframes into dictionaries
for country in os.listdir(countriesPatternsPath):
    country = country.split("_")[0]
    df      = pd.read_csv(countriesFolderPath + country + "_PATTERNS_DF.csv").drop("Unnamed: 0", axis = 1)

    countryDict = {}
    for pattLen in list(set(df["Pattern Length"])):
        pattIndexStarts  = list(set(df[df["Pattern Length"] == pattLen]["Pattern Index Start"]))
        countryDict[pattLen] = {}

        for pattIndexStart in pattIndexStarts:
            dfTemp       = df[df["Pattern Length"] == pattLen]
            dfTemp       = dfTemp[dfTemp["Pattern Index Start"] == pattIndexStart]
            temp         = list(dfTemp["Match Country"])
            countryCount = pd.DataFrame.from_dict(Counter(temp), orient = "index")
            countryCount = countryCount[countryCount[list(countryCount)[0]] >= 2]
            countryDict[pattLen][pattIndexStart] = {}

            for countryMatch in list(countryCount.index):
                countriesMatchDF = dfTemp[dfTemp["Match Country"] == countryMatch]
                temp             = list(countriesMatchDF["Match Index Start"])
                matchIndexStarts = pd.DataFrame.from_dict(Counter(temp), orient = "index")
                matchIndexStarts = matchIndexStarts[matchIndexStarts[list(matchIndexStarts)[0]] >= 2]

                for matchIndexStart in list(matchIndexStarts.index):
                    countriesMatchIndexStartDF = countriesMatchDF[countriesMatchDF["Match Index Start"] == matchIndexStart]
                    countryDict[pattLen][pattIndexStart][countryMatch] = countriesMatchIndexStartDF

    with open(countriesFolderPath + country + "_REFINED_DICT.pkl", "wb") as f:
        pickle.dump(countryDict, f)

Execution times
1. 20m 49.4s